## Contour and Outline Extraction

This notebook shows the steps to extract the outlines from a set of images using the Segment Anything algorithm (Kirillov et al. 2023 and a marching squares algorithm (Lorensen & Cline 1987, Maple 2003). 

### Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import tqdm
from itertools import zip_longest
from pathlib import Path
from copy import deepcopy
from skimage import measure, io
from skimage.filters import gaussian, threshold_otsu
from segment_anything import sam_model_registry, SamPredictor
from SqST_Functions import bb_contour, coords_corresp, rescale

### Data preparation

Set the paths needed:
1. The input data expects an index file. The index file is expected to be a data frame with the columns "Index", "Filename", "Family", "Genus", and "Species".
2. The base_path provides the image folder
3. The out_path determines the saving location for all output
4. Checkpoint_path provides the path to the used segment anything model

In [ ]:
base_path = 'PATH' # Insert path for the folder with your images
out_path = 'PATH' # Insert path for the designated output folder for CSVs
checkpoint_path = 'PATH' # Insert path to the chosen Segment Anything model
index = pd.read_csv('PATH') # Insert path to the index file for the dataset

### Contour Extraction

Collect image paths.

In [ ]:
all_image_paths = []
all_filenames = []

for file in os.listdir(base_path):
	if file == '.DS_Store': # Just necessary if running with Mac
		continue
	all_image_paths.append(os.path.join(base_path, file))
	all_filenames.append(file)


print(str(len(all_image_paths)) + ' images in total')
total_images = len(all_image_paths)

Start contour extraction with a standardised number of coordinates per outline

In [ ]:
all_contours = []
number_of_points = 100   

for i,path in tqdm.tqdm(enumerate(all_image_paths)):
    file = all_filenames[i]
    ID = index[index['Filename']==all_filenames[i]]['Index'].iloc[0]
    image_grey = io.imread(path,as_gray=True)
    xs, ys, cropped_img= bb_contour(path, checkpoint_path)
    if xs is None or cropped_img is None:
            print('No Contour for' +file+ '. Skipping.')
            continue
    
    all_contours.append([[ID]*number_of_points,[file]*number_of_points, xs,ys])

print('All Contours extracted!')

Save contours as a Pandas DataFrame

In [ ]:
contours_table = pd.DataFrame()
rows = []
for idxs, files, xs, ys in all_contours:
    # make each piece iterable
    idxs = np.atleast_1d(idxs)
    files = np.atleast_1d(files)
    xs = np.atleast_1d(xs)
    ys = np.atleast_1d(ys)

    for i, f, x, y in zip_longest(idxs, files, xs, ys, fillvalue=np.nan):
        rows.append((i, f, float(x) if x is not np.nan else np.nan,
                          float(y) if y is not np.nan else np.nan))

contours_table = pd.DataFrame(rows, columns=['Index','Filename','x','y'])
contours_table.to_csv(out_path +'YOUR_FILENAME.csv',index=False)

### Create correspondents between coordinates

For certain shape analysis methods, like eigenshape analysis the coordinates of the outlines need to be corresponding. Therefore the tip of the tooth crown will be defined as the first coordinate.

In [ ]:
groups = np.unique(list(contours_table['Filename']))
filenames = []
all_x = []
all_y = []

for file in groups:
    x = list(contours_table[contours_table['Filename']==file]['x'])
    y = list(contours_table[contours_table['Filename']==file]['y'])
    new_x_, new_y_ = coords_corresp(x,y)
    new_x, new_y = rescale(new_x_, new_y_)
    all_x.extend(new_x)
    all_y.extend(new_y)
    filenames.extend([file]*len(new_x))

cor_coord = pd.DataFrame([filenames, all_x, all_y]).T
cor_coord = cor_coord.rename(columns={0: 'Filename', 1: 'x', 2: 'y'})
cor_coord.to_csv(out_path + 'YOUR_FILENAME.csv', index=False)

Note that this is not needed for the square-root velocity framework.